In [1]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn xgboost scipy matplotlib seaborn plotly nbformat kaleido


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Adaptive ML Pipeline: Rossmann Store Sales Forecasting

## Project Overview
In production Machine Learning, models degrade over time as real-world data shifts. This project implements an **Adaptive ML Pipeline** that monitors for data drift and conditionally retrains the model only when necessary.

### Business Problem Solved
1. **Concept Drift:** Customer behavior changes over time (e.g., "Promo fatigue").
2. **Seasonality Whipsaw:** The model must correctly predict massive December Christmas spikes and immediately return to normal January predictions *without* requiring a dedicated December retraining cycle.
3. **Compute Efficiency:** Blindly retraining every month (Naive approach) wastes compute. Never retraining (Static approach) loses accuracy. This pipeline finds the optimal middle ground.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [3]:
# Loading datasets 'train.csv' and 'store.csv'
train_df = pd.read_csv('data/train.csv', parse_dates=['Date'])
store_df = pd.read_csv('data/store.csv')

# Merging datasets
df = pd.merge(train_df, store_df, on='Store', how='left')

print(f"Initial dataset shape: {df.shape}")
df.head()

Initial dataset shape: (1017209, 18)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [4]:
# 1. Filter out closed stores
df = df[df['Open'] == 1].copy()
print(f"Shape after filtering Open==0: {df.shape}")

# 2. Handle Missing Values
# CompetitionDistance: Filling it with max value
df['CompetitionDistance'] = df['CompetitionDistance'].fillna(78000)

# CompetitionOpenSinceMonth/Year: Filling it with 0 (meaning no competitor opened ever)
df['CompetitionOpenSinceMonth'] = df['CompetitionOpenSinceMonth'].fillna(0)
df['CompetitionOpenSinceYear'] = df['CompetitionOpenSinceYear'].fillna(0)

# Promo2SinceWeek/Year: Filling it with 0
df['Promo2SinceWeek'] = df['Promo2SinceWeek'].fillna(0)
df['Promo2SinceYear'] = df['Promo2SinceYear'].fillna(0)

# PromoInterval: Filling it with empty string
df['PromoInterval'] = df['PromoInterval'].fillna('')

Shape after filtering Open==0: (844392, 18)


In [5]:
# 1. Temporal Features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Is_Weekend'] = (df['DayOfWeek'] >= 5).astype(int)

# 2. The "December" Solver: Explicitly flag December to capture Christmas spikes
df['Is_December'] = (df['Month'] == 12).astype(int)

# 3. StateHoliday Encoding (One-Hot)
# '0' = no holiday, 'a' = public holiday, 'b' = Easter, 'c' = Christmas
df['StateHoliday_a'] = (df['StateHoliday'] == 'a').astype(int)
df['StateHoliday_b'] = (df['StateHoliday'] == 'b').astype(int)
df['StateHoliday_c'] = (df['StateHoliday'] == 'c').astype(int)

# 4. Promo2 Active Check
# Check if the current month is in the store's PromoInterval string
def is_promo2_active(row):
    if row['Promo2'] == 0:
        return 0
    month_abbr = datetime(2000, row['Month'], 1).strftime('%b')
    return 1 if month_abbr in row['PromoInterval'] else 0

df['Is_Promo2_Active'] = df.apply(is_promo2_active, axis=1)

# Define final feature list
feature_cols = [
    'Store', 'DayOfWeek', 'Month', 'Day', 'Year', 'Is_Weekend', 'Is_December',
    'Promo', 'StateHoliday_a', 'StateHoliday_b', 'StateHoliday_c', 'SchoolHoliday',
    'StoreType', 'Assortment', 'CompetitionDistance',
    'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear',
    'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'Is_Promo2_Active'
]

# Convert categoricals to category type for XGBoost
for col in ['StoreType', 'Assortment']:
    df[col] = df[col].astype('category')

target_col = 'Sales'

In [6]:
# Baseline Data (2013)
baseline_df = df[df['Date'] < '2014-01-01'].copy()

# Simulation Data (2014 to mid-2015)
simulation_df = df[df['Date'] >= '2014-01-01'].copy()

# Group simulation data by Month for the loop
simulation_df['YearMonth'] = simulation_df['Date'].dt.to_period('M')
monthly_batches = [group for _, group in simulation_df.groupby('YearMonth')]

print(f"Baseline data shape: {baseline_df.shape}")
print(f"Simulation data shape: {simulation_df.shape}")
print(f"Number of monthly batches to simulate: {len(monthly_batches)}")

Baseline data shape: (337943, 27)
Simulation data shape: (506449, 28)
Number of monthly batches to simulate: 19


In [7]:
import pandas as pd
from scipy import stats

# These features will ALWAYS show drift when comparing a single month to a full year.
drift_feature_cols = [col for col in feature_cols if col not in [
    'Store', 'Month', 'Year', 'Day', 'DayOfWeek', 'Is_December', 'Is_Weekend'
]]

def detect_feature_drift(reference_data, current_data, p_threshold=0.001):
    """
    Compares feature distributions. Uses strict p-value AND effect size.
    """
    drift_detected = False
    drift_details = {}

    for col in drift_feature_cols:
        ref_col = reference_data[col].dropna()
        cur_col = current_data[col].dropna()

        if len(ref_col) == 0 or len(cur_col) == 0:
            continue

        # Identify numeric types (catches int32)
        if pd.api.types.is_numeric_dtype(reference_data[col]):
            # Continuous Features : KS test
            stat, p_value = stats.ks_2samp(ref_col, cur_col)
            # Require strict p-value AND meaningful effect size (5% shift)
            if p_value < p_threshold and stat > 0.05:
                drift_detected = True
                drift_details[col] = {'p_val': p_value, 'effect_size': stat}

        else:
            # Categorical features: Chi-Square test
            categories = pd.Categorical(pd.concat([ref_col, cur_col])).categories
            ref_counts = pd.Series(ref_col).value_counts().reindex(categories, fill_value=0)
            cur_counts = pd.Series(cur_col).value_counts().reindex(categories, fill_value=0)

            if ref_counts.sum() == 0 or cur_counts.sum() == 0:
                continue

            stat, p_value, _, _ = stats.chi2_contingency([ref_counts, cur_counts])
            # For categorical, use a much stricter p-value to avoid false positives from large N
            if p_value < 0.0001:
                drift_detected = True
                drift_details[col] = {'p_val': p_value, 'effect_size': stat}

    return drift_detected, drift_details

In [8]:
# --- Configuration ---
ROLLING_WINDOW_MONTHS = 12
EMA_ALPHA = 0.3 # Smoothing factor for the dynamic error threshold
ERROR_TOLERANCE = 1.20 # Trigger retrain if error is 20% higher than the EMA

# --- 1. Train Baseline Models ---
print("Training Baseline Models now.")
X_baseline = baseline_df[feature_cols]
y_baseline = baseline_df[target_col]

model_static = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
model_static.fit(X_baseline, y_baseline)

model_adaptive = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
model_adaptive.fit(X_baseline, y_baseline)

model_naive = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
model_naive.fit(X_baseline, y_baseline)

# Calculate Initial Baseline MAE to seed the EMA
baseline_predictions = model_static.predict(X_baseline)
initial_baseline_mae = mean_absolute_error(y_baseline, baseline_predictions)
print(f"Initial Baseline MAE (2013): {initial_baseline_mae:.2f}")

# --- Tracking Lists ---
history_mae_static = []
history_mae_adaptive = []
history_mae_naive = []
history_dynamic_threshold = []
retraining_events = []
months_list = []

# Initialize rolling history and the EMA
rolling_history = baseline_df.copy()
ema_mae = initial_baseline_mae

print("Starting Monthly Simulation Loop now.\n")

# --- 2. Simulation Loop ---
for i, batch in enumerate(monthly_batches):
    current_month = batch['YearMonth'].iloc[0]
    months_list.append(str(current_month))

    X_curr = batch[feature_cols]
    y_curr = batch[target_col]

    # A. PREDICT
    pred_static = model_static.predict(X_curr)
    pred_adaptive = model_adaptive.predict(X_curr)
    pred_naive = model_naive.predict(X_curr)

    # B. EVALUATE
    mae_static = mean_absolute_error(y_curr, pred_static)
    mae_adaptive = mean_absolute_error(y_curr, pred_adaptive)
    mae_naive = mean_absolute_error(y_curr, pred_naive)

    history_mae_static.append(mae_static)
    history_mae_adaptive.append(mae_adaptive)
    history_mae_naive.append(mae_naive)

    # C. DYNAMIC ERROR THRESHOLD
    # Calculate the threshold based on the EMA of recent performance
    dynamic_error_threshold = ema_mae * ERROR_TOLERANCE
    history_dynamic_threshold.append(dynamic_error_threshold)

    # Check if current error is significantly higher than the recent "normal"
    is_error_high = mae_adaptive > dynamic_error_threshold

    # D. DETECT DRIFT
    is_drifted, drift_info = detect_feature_drift(baseline_df, batch)

    # E. UPDATE ROLLING HISTORY (Prevents unbounded memory growth)
    rolling_history = pd.concat([rolling_history, batch]).sort_values('Date')
    cutoff_date = rolling_history['Date'].max() - pd.DateOffset(months=ROLLING_WINDOW_MONTHS)
    rolling_history = rolling_history[rolling_history['Date'] > cutoff_date]

    X_roll = rolling_history[feature_cols]
    y_roll = rolling_history[target_col]

    # F. CONDITIONAL RETRAINING (Adaptive Model)
    if is_error_high and is_drifted:
        retraining_events.append(str(current_month))

        model_adaptive = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
        model_adaptive.fit(X_roll, y_roll)

        # Reset the EMA to the NEW model's actual error on this same month,
        post_retrain_preds = model_adaptive.predict(X_curr)
        post_retrain_mae = mean_absolute_error(y_curr, post_retrain_preds)
        ema_mae = post_retrain_mae
    else:
        ema_mae = (EMA_ALPHA * mae_adaptive) + ((1 - EMA_ALPHA) * ema_mae)

    # H. NAIVE RETRAINING (Always retrains on the fresh rolling window)
    model_naive = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
    model_naive.fit(X_roll, y_roll)

print("Simulation Complete!")
print(f"Total Retraining Events for Adaptive Model: {len(retraining_events)} out of {len(monthly_batches)} months.")
if retraining_events:
    print(f"Retraining occurred in: {', '.join(retraining_events)}")

Training Baseline Models now.
Initial Baseline MAE (2013): 885.26
Starting Monthly Simulation Loop now.

Simulation Complete!
Total Retraining Events for Adaptive Model: 2 out of 19 months.
Retraining occurred in: 2014-04, 2014-12


In [16]:
# --- 1. Calculate KPI Metrics for the Dashboard ---
total_months = len(months_list)
adaptive_retrains = len(retraining_events)
naive_retrains = total_months # Naive retrains every month

compute_saved_pct = ((naive_retrains - adaptive_retrains) / naive_retrains) * 100

avg_mae_static = np.mean(history_mae_static)
avg_mae_adaptive = np.mean(history_mae_adaptive)
avg_mae_naive = np.mean(history_mae_naive)

# --- 2. Create Subplot Layout ---
fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"colspan": 2}, None],
           [{}, {}]],
    subplot_titles=(
        "Model Performance (MAE) Over Time",
        "Retraining Frequency (Compute Cost)",
        "Average Error (MAE) Comparison"
    ),
    vertical_spacing=0.18,
    row_heights=[0.6, 0.4]
)

# --- 3. Top Chart: MAE Over Time ---
fig.add_trace(go.Scatter(
    x=months_list, y=history_mae_static,
    mode='lines+markers', name='Static Model (No Retraining)',
    line=dict(color='royalblue', width=2), marker=dict(size=6)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=months_list, y=history_mae_adaptive,
    mode='lines+markers', name='Adaptive Model (Smart Retraining)',
    line=dict(color='forestgreen', width=3), marker=dict(size=8, symbol='square')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=months_list, y=history_mae_naive,
    mode='lines+markers', name='Naive Model (Blind Monthly Retraining)',
    line=dict(color='firebrick', width=2, dash='dash'), marker=dict(size=6, symbol='triangle-up')
), row=1, col=1)

# Plot the Dynamic EMA Threshold instead of a static horizontal line
fig.add_trace(go.Scatter(
    x=months_list, y=history_dynamic_threshold,
    mode='lines', name='Dynamic Error Threshold (EMA)',
    line=dict(color='gray', width=2, dash='dot')
), row=1, col=1)

# Add Vertical Lines for Retraining Events
for event in retraining_events:
    fig.add_vline(x=event, line_dash="solid", line_color="forestgreen", line_width=1, opacity=0.5, row=1, col=1)
    fig.add_annotation(x=event, y=max(history_mae_adaptive)*1.02, text="Retrain",
                       showarrow=False, font=dict(color="forestgreen", size=10), row=1, col=1)

# --- 4. Bottom Left Chart: Retraining Frequency ---
fig.add_trace(go.Bar(
    x=['Naive Model', 'Adaptive Model'],
    y=[naive_retrains, adaptive_retrains],
    text=[f'{naive_retrains} times', f'{adaptive_retrains} times'],
    textposition='auto',
    marker_color=['firebrick', 'forestgreen'],
    showlegend = False
), row=2, col=1)

fig.update_yaxes(title_text="Number of Retrains", row=2, col=1)

# --- 5. Bottom Right Chart: Average MAE ---
fig.add_trace(go.Bar(
    x=['Static', 'Adaptive', 'Naive'],
    y=[avg_mae_static, avg_mae_adaptive, avg_mae_naive],
    text=[f'{avg_mae_static:.0f}', f'{avg_mae_adaptive:.0f}', f'{avg_mae_naive:.0f}'],
    textposition='auto',
    marker_color=['royalblue', 'forestgreen', 'firebrick'], 
    showlegend = False
), row=2, col=2)

fig.update_yaxes(title_text="Average MAE", row=2, col=2)

# --- 6. Add KPI Cards as Annotations at the very top ---
fig.add_annotation(
    x=0.02, y=0.92, xref="paper", yref="paper",
    text=f"<b>Total Months Simulated:</b> {total_months} | "
         f"<b>Adaptive Retrains:</b> {adaptive_retrains} | "
         f"<b>Compute Saved:</b> {compute_saved_pct:.1f}%",
    showarrow=False, font=dict(size=12, color="black"),
    align="center",
    bgcolor="rgba(255,255,255,0.8)"
)

# --- 7. Final Layout Adjustments ---
fig.update_layout(
    title_text="<b>Adaptive ML Pipeline: MLOps Performance Dashboard</b>",
    title_x=0.5,
    title_y=0.95,
    height=800,
    margin=dict(t=150, b=100, l=80, r=50),
    template="plotly_white",
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=0.98)
)

fig.update_xaxes(tickangle=-45)

# Show the dashboard
fig.show()

In [20]:
fig.write_image("assets/dashboard.png", width=1400, height=900, scale=2)